# Metamorphic Testing of the NESTML AMAT Neuronal Model 

Scenario 1: Increase of I(e) leads to increase of synchrony? 

https://nestml.readthedocs.io/en/latest/tutorials/gl_model/gl_model_tutorial.html

In [5]:
import hypothesis 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nest
import pynestml

print("Hypothesis version:", hypothesis.__version__)
print("NEST version:", nest.__version__)



Hypothesis version: 6.163.0
NEST version: 3.9.0-post0.dev14


In [1]:
def evaluate_neuron(neuron_name, module_name, nneurons=1, neuron_parms=None, stimulus_type="constant", poisson_fr=0.0,
                    mu=500., sigma=0., t_sim=300., plot=False, rseed=1000, dt=0.1, input_freq=0.0):
    """
    Run a simulation in NEST for the specified neuron. Inject a stepwise
    current and plot the membrane potential dynamics and spikes generated.
    """
    nest.ResetKernel()
    NESTTools.set_nest_verbosity("ERROR")
    nest.print_time = False
    nest.Install(module_name)
    nest.SetKernelStatus({'rng_seed': rseed,
                          'resolution': dt})
    neuron = nest.Create(neuron_name, nneurons)
    if neuron_parms:
        for k, v in neuron_parms.items():
            nest.SetStatus(neuron, k, v)
    nest.SetStatus(neuron,{'V_m':np.random.uniform(-65.0, -50.0)})

    if stimulus_type == "noise":
        # Create a noise generator
        noise = nest.Create("noise_generator")
        # Set the parameters of the noise generator
        noise_params = {"mean": mu,
                        "std": sigma,
                        "dt": dt,
                        "frequency":input_freq}
        nest.SetStatus(noise, noise_params)
        nest.Connect(noise, neuron)

    elif stimulus_type == "poisson_spikes":
        # Create a Poisson generator device
        poisson = nest.Create("poisson_generator", params={"rate": poisson_fr})

        # Create a parrot neuron
        parrot = nest.Create("parrot_neuron")

        # Connect the Poisson generator to the parrot neuron
        nest.Connect(poisson, parrot)

        # Connect the parrot neuron to each neuron
        nest.Connect(parrot, neuron)
    else:
        assert stimulus_type == "constant"
        nest.SetStatus(neuron, "I_e", mu)

    multimeter = nest.Create("multimeter")
    multimeter.set({"record_from": ["V_m"],
                    "interval": dt})
    spike_recorder = nest.Create("spike_recorder")
    nest.Connect(multimeter, neuron)
    nest.Connect(neuron, spike_recorder)

    nest.Simulate(t_sim)

    dmm = nest.GetStatus(multimeter)[0]
    Voltages = dmm["events"]["V_m"]
    tv = dmm["events"]["times"]

    dSD = nest.GetStatus(spike_recorder, keys="events")[0]
    ns = dSD["senders"]
    ts = dSD["times"]

    _idx = [np.argmin((tv - spike_time)**2) - 1 for spike_time in ts]
    V_m_at_spike_times = Voltages[_idx]

    if plot:
        fig, ax = plt.subplots()
        ax.plot(tv, Voltages)
        ax.scatter(ts, V_m_at_spike_times)
        ax.set_xlabel("Time [ms]")
        ax.set_ylabel("V_m [mV]")
        ax.grid()

    return ts, ns

dt = 0.1
nneurons = 50

ts_const_input, ns_const = evaluate_neuron(neuron_model_name,
                                     module_name,
                                     nneurons=nneurons,
                                     neuron_parms=params,
                                     stimulus_type="constant",
                                     mu=550.,
                                     t_sim=500.0,
                                     dt=dt,
                                     poisson_fr=0.0)

ts_poisson_input, ns_noise = evaluate_neuron(neuron_model_name,
                                     module_name,
                                     nneurons=nneurons,
                                     neuron_parms=params,
                                     stimulus_type="poisson_spikes",
                                     t_sim=500.0,
                                     poisson_fr=2000.0)

# rate averaging parameters
t_start = 0.0
t_stop = 500.0
t_step = 5.0
t_bins = np.arange(t_start, t_stop + t_step, t_step)

# Calculate the average rate
rate_const_input, _ = np.histogram(ts_const_input, bins=t_bins)
rate_const_input = rate_const_input / nneurons / (t_step * 1e-3)      # per trial, per second

rate_poisson_input, _ = np.histogram(ts_poisson_input, bins=t_bins)
rate_poisson_input = rate_poisson_input / nneurons / (t_step * 1e-3)  # per trial, per second

min_rate = min(np.amin(rate_const_input), np.amin(rate_poisson_input))
max_rate = max(np.amax(rate_const_input), np.amax(rate_poisson_input))

plt.figure(figsize=(10,4))

# Raster plots
plt.subplot2grid((3,2),(0,0), rowspan=2)
plt.plot(ts_const_input, ns_const, ".")
plt.xlim(100, 500)
plt.ylabel("#Trial")
plt.title("Constant input current")

plt.subplot2grid((3,2),(0,1), rowspan=2)
plt.plot(ts_poisson_input, ns_noise, ".")
plt.xlim(100, 500)
plt.ylabel("#Trial")
plt.title("Frozen Poisson input")

# Firing rate plots
plt.subplot2grid((3,2),(2,0), rowspan=1)
plt.plot(t_bins[:-1], rate_const_input)
plt.xlabel("Time [ms]")
plt.ylabel("Firing rate [spikes/s]")
plt.xlim(100, 500)
plt.ylim(min_rate, max_rate)

plt.subplot2grid((3,2),(2,1), rowspan=1)
plt.plot(t_bins[:-1], rate_poisson_input)
plt.xlabel("Time [ms]")
plt.ylabel("Firing rate [spikes/s]")
plt.xlim(100, 500)
plt.ylim(min_rate, max_rate)
plt.tight_layout()



NameError: name 'neuron_model_name' is not defined